In [ ]:
# Simple LLM chain without LLMChain class (this was the way used at starting before APIs)

from langchain.llms import OpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate


# load_dotenv() # not required as this is reqd to load api_key when using OpenAI api, here directly using model

llm = OpenAI(model_name='gpt-3.5-turbo', temperature=0.7)

prompt = PromptTemplate(
    input_variables = ["topic"],
    template = "Suggest a catchy blog title about {topic}"
)

# define the output
topic = input('enter a topic')

# format the prompt manually using PromptTemplate
formatted_prompt = prompt.format(topic=topic)

# call the LLM directly
blog_title = llm.predict(formatted_prompt)

# print the output
print("Generated Blog Title:", blog_title)




In [ ]:
# via LLMChain
from langchain.llms import OpenAI
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate


# load_dotenv() # not required as this is reqd to load api_key when using OpenAI api, here directly using model

llm = OpenAI(model_name='gpt-3.5-turbo', temperature=0.7)

prompt = PromptTemplate(
    input_variables = ["topic"],
    template = "Suggest a catchy blog title about {topic}"
)

# create an LLM Chain
chain = LLMChain(llm=llm, prompt=prompt)

# run the chain with aspecific topic
topic = input('enter a topic')
output = chain.run(topic)

# print the output
print("Generated Blog Title:", output)


In [ ]:
# PDF Reader (Without LLMChain class)

from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.llms import OpenAI

# load the document
loader = TextLoader("docs.txt") # ensure docs.txt exists
documents = loader.load()

# Split the text into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

# Convert text into embeddings & store in FAISS
vectorstore = FAISS.from_document(docs, OpenAIEmbeddings())

# Create a retriever (fetches releveant documents)
retriever = vectorstore.as_retriever()

# Manually Retrieve Relevant Documents
query = "What are the key takeways from the documents?"
retrieved_docs = retriever.get_relevant_documents(query)

# Combine Retrieved Text into a Single Prompt
retrieved_text = "\n".join([doc.page_content for doc in retrieved_docs])

# Initialize the LLM
llm = OpenAI(model_name="gpt-3.5-turbo", temperature=0.7)

# Manually Pass Retrieved Text to LLM
prompt = f"Based on the following text, answer the question: {query}\n\n {retrieved_text}"
answer = llm.predict(prompt)

# print the answer
print("Answer:", answer)

In [ ]:
# langchain aam zindegi
class NakliLLM:
    def __init__(self):
        print('LLM created')

    def predict(self, prompt):
        response_list = [
            'Delhi is the capital of India',
            'IPL is a cricket leage',
            'AI stands for Artificial Intelligence'
        ]
        return {'response': random.choice(response_list)}

In [ ]:
llm = NakliLLM()

In [ ]:
llm.predict('What is capital of India?')

In [ ]:
class NakliPromptTemplate:
    def __init__(self):
        self.template = template
        self.input_variables = input_variables

    def format(self):
        return self.template.format(**self.input_variables)

In [ ]:
template = NakliPromptTemplate(
    template = 'Write a {length} poem about {topic}',
    input_variables = ['length', 'topic']
)

In [ ]:
prompt = template.format({'length':'short', 'topic': 'India'})
print(prompt)

In [ ]:
llm.predict(prompt)

In [ ]:
class NakliLLMChain:
    def __init__(self):
        self.llm = llm
        self.prompt = prompt

    def run(self, input_dict):
        final_prompt = self.prompt.format(input_dict)
        result = self.llm.predict(final_prompt)
        return result['response']

In [ ]:
template = NakliPromptTemplate(
    template = 'Write a {length} poem about {topic}',
    input_variables = ['length', 'topic']
)

In [ ]:
chain = NakliLLMChain(llm, template)
chain.run({'length':'short', 'topic': 'India'})

In [ ]:
# standardize
from abc import ABC, abstractmethod

class Runnable(ABC):
    @abstractmethod
    def invoke(input_data):
        pass


class NakliLLM(Runnable):
    def __init__(self):
        print('LLM created')

    def invoke(self, prompt):
        response_list = [
            'Delhi is the capital of India',
            'IPL is a cricket leage',
            'AI stands for Artificial Intelligence'
        ]
        return {'response': random.choice(response_list)}

    def predict(self, prompt):
        response_list = [
            'Delhi is the capital of India',
            'IPL is a cricket leage',
            'AI stands for Artificial Intelligence'
        ]
        return {'response': random.choice(response_list)}


class NakliPromptTemplate(Runnable):
    def __init__(self):
        self.template = template
        self.input_variables = input_variables

    def invoke(self):
        return self.template.format(**self.input_variables)

    def format(self):
        return self.template.format(**self.input_variables)

In [ ]:
llm = NakliLLM() # will throw error if invoke method not implemented

In [ ]:
class NakliStrOutputParser(Runnable):
    def __init__(self):
        pass

    def invoke(self, input_data):
        return input_data['response']

In [ ]:
class RunnableConnector(Runnable):
    def __init__(self, runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_data):
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)

        return input_data


In [ ]:
parser = NakliStrOutputParser()

In [ ]:
chain = RunnableConnector([prompt, llm])
chain.invoke({'length': 'short', 'topic': 'India'})

In [ ]:
chain = RunnableConnector([prompt, llm, parser])
chain.invoke({'length': 'short', 'topic': 'India'})

In [ ]:
# connecting chains 
template1 = NakliPromptTemplate(
    template = 'Write a joke about {topic}',
    input_variables = ['topic']
)

template2 = NakliPromptTemplate(
    template = 'Explain following joke {response}',
    input_variables = ['response']
)

llm = NakliLLM()
parser = NakliStrOutputParser()

chain1 = RunnableConnector([template1, llm])
chain2 = RunnableConnector([template2, llm, parser])

final_chain = RunnableConnector([chain1, chain2])
final_chain.invoke({'topic': 'cricket'})

### RunnableSequence

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain.schema.runnable import RunnableSequence

In [ ]:
load_dotenv()

In [ ]:
prompt1 = PromptTemplate(
    template = 'Write a joke about {topic}',
    input_variables = ['topic']
)

model = ChatOpenAI()

prompt2 = PromptTemplate(
    template = 'Explain the following joke - {text}',
    input_variables = ['text']
)

parser = StrOutputParser()

chain = RunnableSequence(prompt1, model, parser, prompt2, model, parser)

print(chain.invoke({'topic': 'AI'}))

### RunnableParallel

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain.schema.runnable import RunnableSequence, RunnableParallel

load_dotenv()


prompt1 = PromptTemplate(
    template = 'Write a tweet about {topic}',
    input_variables = ['topic']
)

prompt2 = PromptTemplate(
    template = 'Explain the Linkedin joke - {topic}',
    input_variables = ['topic']
)

model = ChatOpenAI()

parser = StrOutputParser()

parallet_chain = RunnableParallel(
    {
        'tweet': RunnableSequence(prompt1, model, parser),
        'linkedin': RunnableSequence(prompt2, model, parser)
    }
)

result = parallet_chain.invoke({'topic': 'AI'})
print(result)

In [ ]:
print(result['tweet'])
print(result['linkedin'])

### RunnablePassthrough

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain.schema.runnable import RunnableSequence, RunnableParallel, RunnablePassthrough

load_dotenv()

passthrough = RunnablePassthrough()

print(passthrough.invoke(2))
print(passthrough.invoke({'name': 'koyel'}))


In [ ]:
prompt1 = PromptTemplate(
    template = 'Write a joke about {topic}',
    input_variables = ['topic']
)

model = ChatOpenAI()

prompt2 = PromptTemplate(
    template = 'Explain the following joke - {text}',
    input_variables = ['text']
)

parser = StrOutputParser()

joke_gen_chain = RunnableSequence(prompt1, model, parser)
parallel_chain = RunnableParallel(
    {
        'joke': RunnablePassthrough(),
        'explanation': RunnableSequence(prompt2, model, parser)
    }
)

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)

print(final_chain.invoke({'topic': 'cricket'}))

In [ ]:
result = final_chain.invoke({'topic': 'cricket'})
print(result['joke'])
print(result['explanation'])

### RunnableLambda

In [ ]:
from langchain.schema.runnable import RunnableLambda

def word_counter(text):
    return len(text.split())

runnable_word_counter = RunnableLambda(word_counter)

print(runnable_word_counter.invoke('Hi there, how are you?'))

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain.schema.runnable import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda

load_dotenv()

def word_counter(text):
    return len(text.split())

prompt = PromptTemplate(
    template = 'Write a joke about {topic}',
    input_variables = ['topic']
)

model = ChatOpenAI()

parser = StrOutputParser()

joke_gen_chain = RunnableSequence(prompt, model, parser)
parallel_chain = RunnableParallel(
    {
        'joke': RunnablePassthrough(),
        'word_count': RunnableLambda(word_count) # or RunnableLambda(lambda x: len(x.split()))
    }
)

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)

print(final_chain.invoke({'topic': 'AI'}))

In [ ]:
result = final_chain.invoke({'topic': 'AI'})
final_result = """{} \n word count - {}""".format(result['joke'], result['word_count'])
print(final_result)

### RunnableBranch

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain.schema.runnable import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda

load_dotenv()


prompt1 = PromptTemplate(
    template = 'Write a detaailed report about {topic}',
    input_variables = ['topic']
)

prompt2 = PromptTemplate(
    template = 'Summarize the following {text}',
    input_variables = ['topic']
)

model = ChatOpenAI()

parser = StrOutputParser()

report_gen_chain = RunnableSequence(prompt1, model, parser)

branch_chain = RunnableBranch(
    (lambda x: len(x.split())>500, RunnableSequence(prompt2, model, parser)),
    RunnablePassthrough()
)

final_chain = RunnableSequence(report_gen_chain, branch_chain)

print(final_chain.invoke({'topic': 'Russia vs Ukraine'}))

### LECEL

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain.schema.runnable import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda

load_dotenv()


prompt1 = PromptTemplate(
    template = 'Write a detaailed report about {topic}',
    input_variables = ['topic']
)

prompt2 = PromptTemplate(
    template = 'Summarize the following {text}',
    input_variables = ['topic']
)

model = ChatOpenAI()

parser = StrOutputParser()

# report_gen_chain = RunnableSequence(prompt1, model, parser)
report_gen_chain = prompt1 | model | parser

# branch_chain = RunnableBranch(
#     (lambda x: len(x.split())>500, RunnableSequence(prompt2, model, parser)),
#     RunnablePassthrough()
# )

branch_chain = RunnableBranch(
    (lambda x: len(x.split())>500, prompt2 | model | parser),
    RunnablePassthrough()
)

final_chain = RunnableSequence(report_gen_chain, branch_chain)

print(final_chain.invoke({'topic': 'Russia vs Ukraine'}))